In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from eda_agent.eda_agent import build_app

app = build_app()
print('그래프 빌드 완료')

그래프 빌드 완료


In [2]:
import json, re, os
from eda_agent.tools.visualize import set_output_dirs

with open("eda_agent_input.json", encoding="utf-8") as f:
    agent_input = json.load(f)

# grain 이름 추출
raw_grain = agent_input.get("mart_design", {}).get("grain", "unknown")
grain_name = re.sub(r"[^a-z0-9]+", "_", raw_grain.lower()).strip("_")

# outputs/ 위치
base_outputs = next((d for d in ['outputs', '../outputs'] if os.path.exists(d)), 'outputs')
os.makedirs(base_outputs, exist_ok=True)

# 다음 실행 번호 자동 결정
existing = [
    d for d in os.listdir(base_outputs)
    if re.match(rf"^{re.escape(grain_name)}_\d+$", d)
    and os.path.isdir(os.path.join(base_outputs, d))
] if os.path.exists(base_outputs) else []
n = max([int(re.search(r"_(\d+)$", d).group(1)) for d in existing], default=0) + 1

grain_dir = os.path.join(base_outputs, f"{grain_name}_{n}")
os.makedirs(grain_dir, exist_ok=True)

set_output_dirs(grain_dir)
print(f"출력 폴더: {grain_dir}")

result = app.invoke({
    "user_question": agent_input["user_question"],
    "target_table":  agent_input["target_table"],
    "mart_design":   agent_input["mart_design"],
    "question_type": agent_input.get("question_type", "comparison"),
    "analysis_plan": {},
    "inspect_result": "",
    "quality_result": "",
    "distribution_result": "",
    "comparison_result": "",
    "relationship_result": "",
    "time_result": "",
    "clustering_result": {},
    "time_columns": [],
    "count_column": "",
    "has_time_column": False,
    "insight_result": "",
    "hypotheses": "",
    "final_summary": "",
    "key_charts": [],
    "statistical_metadata": {},
})
print('=== EDA 완료 ===')

출력 폴더: outputs\seller_level_7


c:\Users\Lisa\Desktop\대학교\대외활동\BOAZ\BOAZ_ADV\EDA_Agent_0427\eda_agent\tools\visualize.py:378: UserWarning: Glyph 53356 (\N{HANGUL SYLLABLE KEU}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
c:\Users\Lisa\Desktop\대학교\대외활동\BOAZ\BOAZ_ADV\EDA_Agent_0427\eda_agent\tools\visualize.py:378: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
c:\Users\Lisa\Desktop\대학교\대외활동\BOAZ\BOAZ_ADV\EDA_Agent_0427\eda_agent\tools\visualize.py:378: UserWarning: Glyph 49353 (\N{HANGUL SYLLABLE SAEG}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
c:\Users\Lisa\Desktop\대학교\대외활동\BOAZ\BOAZ_ADV\EDA_Agent_0427\eda_agent\tools\visualize.py:380: UserWarning: Glyph 53356 (\N{HANGUL SYLLABLE KEU}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight", dpi=120)
c:\Users\Lisa\Desktop\대학교\대외활동\BOAZ\BOAZ_ADV\EDA_Agent_0427\eda_agent\tools\visualize.py:380: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font(s) DejaVu

=== EDA 완료 ===


In [3]:
print(result['final_summary'])

이 데이터는 판매자 성과가 특정 지역에 집중되는 헤드 집중형 구조를 보이며, 특히 MA 지역에서 높은 주문 수와 매출이 관찰된다. 평균 주문 수와 매출 간의 강한 양의 상관관계(0.801)를 검증하기 위해 피어슨 상관검정을 사용할 수 있다. 또한, 평균 리뷰 점수와 늦은 배송 비율 간의 음의 상관관계(-0.433)를 스피어만 상관검정으로 검증할 수 있다. 평균 배송비가 매출에 미치는 영향이 낮다는 가설은 단순선형회귀로 검증 가능하다. 다음 에이전트는 피어슨 상관검정으로 매출과 주문 수 관계를 우선 검증하라.


In [4]:
import json, os, glob

key_charts = result.get("key_charts", [])

all_dir = os.path.join(grain_dir, "all")
all_charts = sorted(glob.glob(os.path.join(all_dir, "*.png")))

eda_output = {
    "user_question":        result["user_question"],
    "analysis_plan":        result["analysis_plan"],
    "insight_result":       result["insight_result"],
    "hypotheses":           result["hypotheses"],
    "final_summary":        result["final_summary"],
    "statistical_metadata": result.get("statistical_metadata", {}),
    "key_charts":           key_charts,
    "all_charts":           all_charts,
}

json_path = os.path.join(grain_dir, "eda_agent_output.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(eda_output, f, ensure_ascii=False, indent=2)

print(f"저장 완료 — {grain_dir}")
print(f"전체 차트 {len(all_charts)}개 / 핵심 차트 {len(key_charts)}개")
print("\nkey_charts:")
for p in key_charts:
    print(" ", os.path.basename(p))

저장 완료 — outputs\seller_level_7
전체 차트 22개 / 핵심 차트 6개

key_charts:
  bubble_total_orders_vs_total_revenue.png
  cluster_profile.png
  cluster_scatter_total_orders_vs_total_revenue.png
  correlation_heatmap.png
  grouped_bar_top_categories.png
  radar_top_categories.png


In [5]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from eda_agent.report import generate_report

report_path = generate_report(json_path, os.path.join(grain_dir, "report.html"))
print(f"리포트 생성 완료: {report_path}")

리포트 생성 완료: outputs\seller_level_7\report.html


In [6]:
# ── 리포트만 다시 생성할 때 이 셀만 실행 ──
# 에이전트를 돌린 세션이면 grain_dir 자동 사용
# 새 세션에서 단독 실행할 때만 아래 target_grain_dir 직접 지정
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from eda_agent.report import generate_report

target_grain_dir = "outputs/category_level_1"  # 새 세션 단독 실행 시에만 수정

_dir = grain_dir if 'grain_dir' in dir() else target_grain_dir
_json = os.path.join(_dir, "eda_agent_output.json")
_html = os.path.join(_dir, "report.html")
print(generate_report(_json, _html))

outputs\seller_level_7\report.html
